In [53]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [54]:
from pathlib import Path
import os
from os.path import join
import sys
import sqlite3
# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *
from locallib.box import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

In [55]:
year = 2026
customer_name = 'Cadent'

In [56]:
#Get the columns of the view and get the KPI definitions
cols = Query("PRAGMA table_info('Weekly_KPI')").execute(KPIHub_Conn)
cols.db.set_query("SELECT * FROM KPI_Definition WHERE name IN (SELECT name from temp_KPI)")
kpi_col = cols.db.execute(KPIHub_Conn, source_col = 'name', temp_table_name = 'temp_KPI')
output_dict = {kpi_col['Name']: [kpi_col['Unit'], kpi_col['Description'], kpi_col['Formula']] for _, kpi_col in kpi_col.iterrows()}


In [57]:
output_dict

{'AssetCoveredLengthKm': ['Km',
  'Total length of all the assets covered in the report',
  ''],
 'AvgSpeedKm': ['Km/h',
  'Average speed counting all the cars in the specified week',
  ''],
 'B0Count': ['Lisa',
  'Total number of B0 in the specified week, (No Disposition 2)',
  ''],
 'B0Density': ['Lisa / Km',
  'Number of B0 per Km of Asset Covered',
  'B0Count / DistributionPipeCoveredKm'],
 'B0Share': ['Percent', 'Number of B0 per Lisa', 'B0Count / LisaCount'],
 'B1Count': ['Lisa',
  'Total number of B1 in the specified week, (No Disposition 2)',
  ''],
 'B1Density': ['Lisa / Km',
  'Number of B1 per Km of Asset Covered',
  'B1Count / DistributionPipeCoveredKm'],
 'B1Share': ['Percent', 'Number of B1 per Lisa', 'B1Count / LisaCount'],
 'Bm1Count': ['Lisa',
  'Total number of B-1 in the specified week, (No Disposition 2)',
  ''],
 'Bm1Density': ['Lisa / Km',
  'Number of B-1 per Km of Asset Covered',
  'Bm1Count / DistributionPipeCoveredKm'],
 'Bm1Share': ['Percent', 'Number of B-1 

In [58]:
#Get total KPI
def week_dates(year, week):
    """
    Returns the start and end dates (Monday to Sunday) of the given ISO week and year.
    """
    # Ensure year and week are integers (convert if they're not)
    year = int(year)
    week = int(week)
    # Use the ISO calendar to get the Monday of the week
    # ISO: Monday is 1, Sunday is 7
    from datetime import date, timedelta
    # Python 3.8+ provides fromisocalendar
    week_start = date.fromisocalendar(year, week, 1)
    week_end = week_start + timedelta(days=6)
    # Make sure week_start is not before 2026-01-01
    min_start = date(2026, 1, 1)
    if week_start < min_start:
        week_start = min_start
        week_end = week_start + timedelta(days=6)
    return week_start, week_end


In [59]:
regions = Query(f"SELECT DISTINCT BoundaryRegion FROM Weekly_KPI WHERE CustomerName = '{customer_name}'").execute(KPIHub_Conn)
kpi_data = Query(f"SELECT * FROM Weekly_KPI WHERE CustomerName = '{customer_name}' AND Year = {year}").execute(KPIHub_Conn)

In [60]:

with pd.ExcelWriter("Cadent_E_KPI.xlsx") as writer:
    # Handle 'regions' DataFrame correctly: iterate over its rows, not directly over DataFrame
    for _, region_row in regions.iterrows():
        region = region_row['BoundaryRegion']
        # Fix the ambiguity with Series comparison
        if pd.isnull(region):
            export_df = kpi_data[kpi_data['BoundaryRegion'].isnull()]
            export_df['PeakAboveSATCount'].fillna(0, inplace=True)
            sheet_name = f"KPI Global {year} Weekly"
        else:
            export_df = kpi_data[kpi_data['BoundaryRegion'] == region]
            export_df['PeakAboveSATCount'].fillna(0, inplace=True)
            sheet_name = f"KPI {region} {year} Weekly"

        # Process the WeekDates
        export_df = export_df.copy()  # Avoid SettingWithCopyWarning
        export_df['WeekDates'] = export_df['PeriodValue'].apply(lambda week: f"{week_dates(2026, int(week))[0]} to {week_dates(2026, int(week))[1]}")
        # Move "WeekDates" to the first column in export_df
        cols = list(export_df.columns)
        if "WeekDates" in cols:
            cols.insert(0, cols.pop(cols.index("WeekDates")))
            export_df = export_df[cols]

        # Process the column name
        units = []
        for col in export_df.columns:
            if col in kpi_col['Name'].values:
                units.append(kpi_col.loc[kpi_col['Name'] == col, 'Unit'].values[0])
            else:
                units.append("")

        # Write the filtered DataFrame to the first sheet
        # Write columns and units as first two rows, then export the rest of the DataFrame
        rows = export_df.values.tolist()
        full_rows = [export_df.columns.tolist(), units] + rows
        temp_df = pd.DataFrame(full_rows)

        temp_df.to_excel(writer, sheet_name=sheet_name, index=False, header=False)

        # Post-process the sheet for bold and center alignment of the first two columns
        worksheet = writer.sheets[sheet_name]
        # Create bold and center formats
        bold_center = writer.book.add_format({'bold': True, 'align': 'center'})
        center = writer.book.add_format({'align': 'center'})

        # The first two rows (headers and units): apply bold and center format to *all* columns, not just columns 0 and 1
        for row_idx in range(2):
            for col_idx in range(len(full_rows[row_idx])):
                worksheet.write(row_idx, col_idx, full_rows[row_idx][col_idx], bold_center)  # overwrite with bold+center

        # All other rows: just center the first two columns
        for i, row in enumerate(rows, start=2):
            for col in range(2):
                worksheet.write(i, col, row[col], center)
        # Add a colored line (cell border) at the bottom of all cells in the second row (units row)
        bottom_border_format = writer.book.add_format({'bottom': 1, 'bottom_color': '#000000', 'align': 'center', 'bold': True})
        for col_idx in range(len(full_rows[1])):
            worksheet.write(1, col_idx, full_rows[1][col_idx], bottom_border_format)
    

    # Write the key and its list of [Unit, Description, Formula Used] as columns in the second sheet
    desc_rows = []
    for key, value in output_dict.items():
        if isinstance(value, list) and len(value) == 3:
            # Value is a list: [Unit, Description, Formula Used]
            row = [key] + value
        else:
            # Fallback in case the dictionary isn't formatted as expected
            row = [key, "", "", ""]
        desc_rows.append(row)
    columns = ["Key", "Unit", "Description", "Formula Used"]
    desc_df = pd.DataFrame(desc_rows, columns=columns)
    desc_df.to_excel(writer, sheet_name="KPI Descriptions", index=False)

#box_obj = BoxFile(local_path = "Cadent_E_KPI.xlsx", box_file_id = 2266637913401)
#box_obj.upload()


/tmp/ipykernel_69938/2796859994.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  export_df['PeakAboveSATCount'].fillna(0, inplace=True)
/tmp/ipykernel_69938/2796859994.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  export_df['PeakAboveSATCount'].fillna(0, inplace=True)
/tmp/ipykernel_69938/2796859994.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  export_df['PeakAboveSATCount'].fillna(0, inplace=True)
/tmp/ipykernel_69938

In [61]:
Query(query = "SELECT * FROM Weekly_KPI WHERE BoundaryRegion IS NULL").execute(KPIHub_Conn)

,Year,PeriodValue,CustomerName,BoundaryRegion,ReportAssetLengthKm,AssetCoveredLengthKm,DistributionPipeKm,DistributionPipeCoveredKm,CumulativeAssetCoveredLengthKm,ServicePipeKm,...,Bm2Density,B0Share,B1Share,Bm1Share,Bm2Share,NGShare,PGShare,Not_NGShare,POR,CurrentCompletion
0,2023,14,Cadent,None,131.74,124.48,0.00,0.00,124.48,0.00,...,NaN,0.23,0.02,0.64,0.12,0.80,0.15,0.05,NaN,NaN
1,2023,15,Cadent,None,139.41,125.50,0.00,0.00,249.98,0.00,...,NaN,0.36,0.01,0.58,0.05,0.86,0.13,0.02,NaN,NaN
2,2023,16,Cadent,None,266.26,246.80,0.00,0.00,496.78,0.00,...,NaN,0.27,0.01,0.67,0.05,0.83,0.12,0.05,NaN,NaN
3,2023,17,Cadent,None,278.67,272.43,0.00,0.00,769.21,0.00,...,NaN,0.23,0.01,0.68,0.08,0.82,0.14,0.04,NaN,NaN
4,2023,19,Cadent,None,74.01,72.07,0.00,0.00,841.29,0.00,...,NaN,0.35,0.01,0.63,0.01,0.66,0.23,0.11,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164,2026,24,Cadent,None,2966.58,2789.11,2894.17,2735.39,46698.48,72.42,...,0.11,0.19,0.01,0.70,0.10,0.79,0.11,0.09,127000.0,37.0
165,2026,25,Cadent,None,3157.42,2979.87,3074.96,2913.74,49678.34,82.46,...,0.15,0.20,0.01,0.69,0.10,0.82,0.09,0.09,127000.0,39.0
166,2026,26,Cadent,None,2897.81,2686.67,2809.89,2618.41,52365.01,87.92,...,0.15,0.19,0.01,0.70,0.10,0.71,0.12,0.17,127000.0,41.0
167,2026,27,Cadent,None,3529.37,3306.08,3440.18,3237.35,55671.10,89.19,...,0.12,0.20,0.01,0.69,0.10,0.75,0.11,0.13,127000.0,44.0
